# 27 — Model B: exploratory analysis on `VALIDATE-dev`

Step 5 of [ABSA_TRAINING_PROPOSAL.md](../ABSA_TRAINING_PROPOSAL.md).

> **This notebook may only ever touch `VALIDATE-dev`.**
>
> `VALIDATE-test` is frozen — it is read exactly once, at the end, by
> `src/absa_eval.py --final`, which logs every read to `ABSA_TEST_READ_LOG`.
> Every configuration choice, threshold, and error-analysis conclusion is
> made *here*, on dev, so that the test partition is never used to choose
> between configurations. Nothing below passes `partition='test'`.

Prerequisites:

```bash
uv run python src/absa_validate.py                                    # the partition
uv run python src/absa_train_b.py  --arm baseline                     # a model
uv run python src/absa_eval.py --predict --arm baseline --partition dev
```

In [ ]:
import sys
from collections import Counter

import duckdb
import pandas as pd

sys.path.insert(0, "../src")

from absa_label import DB_PATH
from absa_model import aggregate_review
from absa_validate import ASPECTS5, CLASSES, load_partition
from absa_eval import load_cached, macro_f1

PARTITION = "dev"   # <- never change this to 'test'
ARM = "baseline"

con = duckdb.connect(str(DB_PATH), read_only=True)
docs = load_partition(con, PARTITION)
print(f"{len(docs):,} VALIDATE-{PARTITION} docs")
print(Counter(d["language"] for d in docs), Counter(d["source"] for d in docs))

## 1. What the reference standard looks like

Includes the `Branding` → `experience` remap, which is already applied — it
happens at load time in `absa_validate.py`, upstream of the split, so dev and
test can never diverge in how the label is treated.

In [ ]:
gold = pd.DataFrame([
    {"doc_id": d["doc_id"], "lang": d["language"], "source": d["source"], **d["labels"]}
    for d in docs
])
dist = pd.concat(
    {a: gold.groupby("lang")[a].value_counts(normalize=True).unstack().fillna(0)
     for a in ASPECTS5},
    names=["aspect"],
)
(dist[CLASSES] * 100).round(1)

In [ ]:
# How many docs the remap touched, and how many of those had no other
# experience span - i.e. where the remap is the ONLY reason experience is
# marked present. That is the population the train/validate taxonomy mismatch
# actually bites on.
con.execute(f"""
    SELECT source,
           count(*) FILTER (WHERE n_branding_spans > 0)              AS branding_docs,
           count(*)                                                   AS docs,
           round(100.0 * count(*) FILTER (WHERE n_branding_spans > 0)
                 / count(*), 1)                                       AS pct
    FROM ABSA_VALIDATE WHERE partition = '{PARTITION}'
    GROUP BY 1 ORDER BY 1
""").df()

## 2. Model B on dev — per aspect, per language

The bilingual claim is *vi ≈ en*. This is where you find out whether it holds,
and it is the number to iterate against.

In [ ]:
cached = load_cached(con, ARM, f"validate_{PARTITION}")
print(f"{len(cached):,} docs with cached sentence predictions")

scored = [d for d in docs if d["doc_id"] in cached]
rows = []
for d in scored:
    for a in ASPECTS5:
        rows.append({
            "doc_id": d["doc_id"], "lang": d["language"], "source": d["source"],
            "aspect": a, "gold": d["labels"][a],
            "pred": aggregate_review(cached[d["doc_id"]], a, "primary"),
            "pred_ruleB": aggregate_review(cached[d["doc_id"]], a, "ruleB"),
            "n_sent": len(cached[d["doc_id"]][a]),
        })
pred = pd.DataFrame(rows)
pred.head()

In [ ]:
f1 = (pred.groupby(["aspect", "lang"])
          .apply(lambda g: macro_f1(list(g["gold"]), list(g["pred"])),
                 include_groups=False)
          .unstack())
f1["all"] = pred.groupby("aspect").apply(
    lambda g: macro_f1(list(g["gold"]), list(g["pred"])), include_groups=False)
f1.loc["MEAN"] = f1.mean()
f1.round(3)

In [ ]:
# Same, per source: TripAdvisor vs Booking is a second cross-platform cut on
# top of the en/vi one.
(pred.groupby(["aspect", "source"])
     .apply(lambda g: macro_f1(list(g["gold"]), list(g["pred"])),
            include_groups=False)
     .unstack().round(3))

## 3. Where it goes wrong — confusion per aspect

`negative` and `neutral` are the expected hard cases: neutral is ~3 % of the
silver training signal, and the reference standard's neutral boundary is one
annotator's judgement (see the κ ceiling in `src/absa_kappa.py`).

In [ ]:
for a in ASPECTS5:
    sub = pred[pred["aspect"] == a]
    cm = pd.crosstab(sub["gold"], sub["pred"]).reindex(
        index=CLASSES, columns=CLASSES).fillna(0).astype(int)
    print(f"\n=== {a} ===  (rows = gold, cols = predicted)")
    print(cm.to_string())

In [ ]:
# Recall of the two hard classes, split by language - the single most useful
# diagnostic for deciding whether class weighting is pulling its weight.
hard = pred[pred["gold"].isin(["negative", "neutral"])]
(hard.assign(hit=lambda df: df["gold"] == df["pred"])
     .groupby(["aspect", "gold", "lang"])["hit"]
     .agg(["mean", "size"]).round(3))

## 4. Aggregation sensitivity, on dev

The reported sensitivity is computed on **test** by `--ablation` (both rules
were fixed in advance, so that is a sensitivity, not a selection). Looking at
it here first is exactly what dev is for.

In [ ]:
agg = pd.DataFrame({
    "primary": pred.groupby("aspect").apply(
        lambda g: macro_f1(list(g["gold"]), list(g["pred"])), include_groups=False),
    "ruleB": pred.groupby("aspect").apply(
        lambda g: macro_f1(list(g["gold"]), list(g["pred_ruleB"])),
        include_groups=False),
})
agg["delta_F1_points"] = (agg["ruleB"] - agg["primary"]) * 100
agg.loc["MEAN"] = agg.mean()
agg.round(3)

In [ ]:
# How often the two rules actually disagree, and in which direction. If they
# rarely disagree the delta above is uninformative rather than reassuring.
d = pred[pred["pred"] != pred["pred_ruleB"]]
print(f"{len(d):,} / {len(pred):,} (aspect, doc) cells differ "
      f"({len(d) / len(pred):.2%})")
pd.crosstab(d["pred"], d["pred_ruleB"])

## 5. Error browser

Read actual documents the model gets wrong. Most "errors" on a single-annotator
reference are worth checking against the text before being treated as model
failures — that is precisely what the κ subset exists to quantify.

In [ ]:
ASPECT, GOLD, PRED, N = "service", "negative", "positive", 5

text = {d["doc_id"]: d["review_text"] for d in scored}
sel = pred[(pred["aspect"] == ASPECT) & (pred["gold"] == GOLD)
           & (pred["pred"] == PRED)]
print(f"{len(sel):,} docs where {ASPECT}: gold={GOLD}, pred={PRED}\n")
for _, r in sel.head(N).iterrows():
    print("=" * 78)
    print(f"{r['doc_id']}  [{r['lang']}]  sentences={r['n_sent']}")
    print(text[r["doc_id"]][:700])
    print(f"  per-sentence {ASPECT}: {cached[r['doc_id']][ASPECT]}")

## 6. Does document length hurt?

Model B is trained on sentences and scored on documents, so a long review
gives the majority vote more chances to drown a single genuine mention. If F1
falls off sharply with sentence count, that is an argument for the learned
pooling noted as future work (SAAM, arXiv:2012.08407) — **not** something to
fix by hand-tuning the rule after seeing test.

In [ ]:
buckets = pd.cut(pred["n_sent"], [0, 1, 2, 4, 8, 16, 1000],
                 labels=["1", "2", "3-4", "5-8", "9-16", "17+"])
(pred.assign(bucket=buckets)
     .groupby(["bucket", "aspect"], observed=True)
     .apply(lambda g: macro_f1(list(g["gold"]), list(g["pred"])),
            include_groups=False)
     .unstack().round(3))

## 7. The reliability ceiling

Until `ABSA_KAPPA` is populated, every number above is **provisional**: all
34,468 reference documents carry one annotation from one annotator, so F1
measures agreement with one individual's judgement rather than with a
validated consensus standard.

In [ ]:
try:
    k = con.execute("""
        SELECT language, aspect, metric, round(kappa, 3) AS kappa, n_docs
        FROM ABSA_KAPPA WHERE metric = 'full4' ORDER BY aspect, language
    """).df()
    display(k if len(k) else "ABSA_KAPPA is empty")
except duckdb.CatalogException:
    print("No ABSA_KAPPA table yet — the double-annotation round has not been\n"
          "scored. Every F1 in this notebook is PROVISIONAL.\n\n"
          "  uv run python src/absa_kappa.py --export     # blind task files\n"
          "  ... second annotator works ...\n"
          "  uv run python src/absa_kappa.py --score <their export>.json")

In [ ]:
con.close()